# Persian isolated letter recognizer

This notebook uses the YOLO11n checkpoint to predict isolated Persian letters in image datasets. It can pull a public GitHub repository or accept a ZIP uploaded directly in Colab. It writes a CSV of predictions for download.

### Use a GitHub dataset

Upload your image files or a ZIP into the `dataset/` folder of a public GitHub repository, then set `DATASET_REPO` in the configuration cell. The default is this project repository. The notebook clones that repository and scans `dataset/` recursively. Organize images into class-named folders if you want the folder names included as reference labels in the CSV.

### Or upload a ZIP in Colab

Set `DATASET_SOURCE = "colab_upload"` and run the upload cell. Upload a ZIP containing the images (or upload individual images).

The model was trained on synthetic isolated typography. Camera photos and natural handwriting have not been validated.

In [ ]:
%pip -q install ultralytics==8.4.172 pi-heif
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import json, subprocess, sys, torch
from ultralytics import YOLO

PACKAGE_DIR = Path("/content/persian_ocr_package")
if not PACKAGE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yt22-orb/persian-ocr.git", str(PACKAGE_DIR)], check=True)
sys.path.insert(0, str(PACKAGE_DIR))
from persian_ocr.crop_cnn import CropLetterCNN, predict_crop_pipeline

matches = list(Path("/content/drive/MyDrive").rglob("yolo11n.pt"))
if not matches:
    raise FileNotFoundError("Add the published yolo11n.pt from the latest release to My Drive, then rerun this cell.")
MODEL_PATH = matches[0]
device = "cuda" if torch.cuda.is_available() else "cpu"
detector = YOLO(str(MODEL_PATH))
labels = json.loads((PACKAGE_DIR / "configs/labels.json").read_text())
policy = json.loads((PACKAGE_DIR / "configs/crop_pipeline_decision.json").read_text())
classifier = CropLetterCNN(dropout=0.25, classes=len(labels)).to(device)
checkpoint = torch.load(PACKAGE_DIR / "artifacts/crop_cnn/best.pt", map_location=device, weights_only=True)
classifier.load_state_dict(checkpoint["state_dict"])
classifier.eval()
print("Loaded detector:", MODEL_PATH)
print("Loaded calibrated crop classifier; device:", device)


In [ ]:
# Choose "github" or "colab_upload". For GitHub, upload data under dataset/ in a public repository.
DATASET_SOURCE = "github"
DATASET_REPO = "yt22-orb/persian-ocr"  # owner/repository
DATASET_BRANCH = "main"
DATASET_SUBDIR = "dataset"
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}


In [ ]:
from pathlib import Path
import shutil, subprocess, zipfile
from google.colab import files

DATASET_DIR = Path("/content/persian_ocr_dataset")
if DATASET_SOURCE == "github":
    clone_dir = Path("/content/persian_ocr_dataset_repo")
    if clone_dir.exists():
        shutil.rmtree(clone_dir)
    repo_url = f"https://github.com/{DATASET_REPO}.git"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", DATASET_BRANCH, repo_url, str(clone_dir)], check=True)
    DATASET_DIR = (clone_dir / DATASET_SUBDIR).resolve()
    if not DATASET_DIR.is_relative_to(clone_dir.resolve()):
        raise ValueError("DATASET_SUBDIR must stay inside the cloned repository.")
    if not DATASET_DIR.exists():
        raise FileNotFoundError(f"No {DATASET_SUBDIR}/ folder in {DATASET_REPO}. Upload images or a ZIP there.")
elif DATASET_SOURCE == "colab_upload":
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    for filename, content in uploaded.items():
        source = Path("/content") / Path(filename).name
        source.write_bytes(content)
        if source.suffix.lower() == ".zip":
            with zipfile.ZipFile(source) as archive:
                base = DATASET_DIR.resolve()
                for entry in archive.infolist():
                    destination = (DATASET_DIR / entry.filename).resolve()
                    if destination != base and base not in destination.parents:
                        raise ValueError(f"Unsafe path in ZIP: {entry.filename}")
                archive.extractall(DATASET_DIR)
        else:
            shutil.copy2(source, DATASET_DIR / source.name)
else:
    raise ValueError("DATASET_SOURCE must be github or colab_upload")
image_paths = sorted(p for p in DATASET_DIR.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
if not image_paths:
    raise FileNotFoundError(f"No supported images found under {DATASET_DIR}")
print(f"Found {len(image_paths)} images in {DATASET_DIR}")

In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image

rows = []
for image_path in image_paths:
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    result = predict_crop_pipeline(detector, classifier, image, policy, device=device)
    pred = result.get("prediction")
    class_id = pred["class_id"] if pred else None
    confidence = pred["confidence"] if pred else None
    accepted = result["accepted"]
    relative_path = image_path.relative_to(DATASET_DIR)
    expected = relative_path.parts[0] if len(relative_path.parts) > 1 else ""
    rows.append({"file": str(relative_path), "folder_label": expected,
                 "predicted_letter": labels[class_id]["letter"] if accepted else "",
                 "transliteration": labels[class_id]["transliteration"] if accepted else "",
                 "confidence": confidence, "status": "accepted" if accepted else "review",
                 "reason": result["reason"],
                 "detections": len(result.get("detector", {}).get("candidates", []))})
results = pd.DataFrame(rows)
display(results.head(20))
csv_path = Path("/content/persian_ocr_predictions.csv")
results.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("Status counts:", results["status"].value_counts().to_dict())
files.download(str(csv_path))
